# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
import pandas as pd  # noqa: F401
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them, and calls the external seeding module.
Author: Juan Martín Carini
Date: 2026-05-11
"""


import src.database  # noqa: E402, F401
from src.database import SessionLocal, engine  # noqa: E402, F401
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

if __name__ == "__main__":
    reset_and_seed()

In [ ]:
import src.database.models as models
import src.portfolio.purchase as purchase
reload(purchase)

paths = {
    "personas": "../data/PERSONAS.CSV",
    "prestamos": "../data/PRESTAMOS.CSV",
    "cuotas": "../data/CUOTAS.CSV"}

NewAssociate = models.SocioComercial()
NewAssociate.create_socio("Mentiritas S.A.", 30713257880)
NewRelation = models.Relacion()
NewRelation.add_single_mapping(1, "provincias", 1, 2)
NewRelation.add_single_mapping(1, "provincias", 3, 6)
NewRelation.add_single_mapping(1, "provincias", 7, 5)
NewRelation.add_single_mapping(1, "provincias", 16, 16)
NewAssociate = models.SocioComercial()
NewAssociate.create_socio("Mutual Uno", 30713257870)
NewAssociate.create_socio("Mutual Dos", 30713257860)
NewAssociate.create_socio("Mutual Tres", 30713157860)
NewRelation.add_single_mapping(1, "socios_comerciales", 2, 14)
NewRelation.add_single_mapping(1, "socios_comerciales", 3, 20)
NewRelation.add_single_mapping(1, "socios_comerciales", 4, 7)
NewFolder_1 = purchase.PortfolioPurchase()
NewFolder_1.process_full_portfolio("Folder Test 1", "2026/05/31", 0.45, 30713257880, "Mentiritas S.A.", paths=None, recurso=False, iva=True)
NewFolder_2 = purchase.PortfolioPurchase()
NewFolder_2.process_full_portfolio("Folder Test 2", "2026/06/03", 0.44, 30713257880, "Mentiritas S.A.", paths=None, recurso=True, iva=False);

In [ ]:
pd.read_sql("carteras", engine)

In [ ]:
import src.logic.collections as collections
reload(collections)

NewColl = collections.CollectionManager()
NewColl.process_resource("PROVEEDOR_CUIT", 30713257880, 4976222.07, "2026/05/30")

In [ ]:
import src.reports.balances as calcular
reload(calcular)

df = calcular.saldos("2026/06/28", con_saldo=False, propias=True, agrupar=True, socios=True, vencimientos=True, recurso=True, iva=True)
# df.loc["Total"] = df.sum()
df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df

In [ ]:
import src.portfolio.sell as sell
reload(sell)

# Inicialización interactiva del manager de ventas
NewSell = sell.PortfolioSell()

# Probamos los nuevos filtros
df_venta = NewSell.fetch_available_installments_for_sale(
    # mora=True,
    # cuotas_completas=True,
    socio_originador_id=[2]
)

display(df_venta[["capital", "interes", "subtotal"]].sum().map("$ {:,.2f}".format))

# Simulamos la ejecución de la venta
cartera_vendida = NewSell.execute_portfolio_sale(
    nombre_cartera='Venta Cartera Test 1',
    fecha_venta='2026/06/04',
    tna_descuento=0.55,
    cuit_comprador='30711111110',
    razon_social_comprador='Fondo Inversor S.A.',
    recurso=False
)

In [ ]:
import src.logic.collections as collections
reload(collections)

NewColl = collections.CollectionManager()
NewColl.process_standard_payment("PROVEEDOR_CUIT", 30713257880, 13043888.66, "2026/06/29")

In [ ]:
reload(calcular)

df = calcular.saldos("2026/06/29", con_saldo=True, propias=False, agrupar=True, vencimientos=True)
totales =  df[["Capital", "Interés", "IVA", "Total"]].sum()
df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)
display(df)
display(totales.map("$ {:,.2f}".format))

In [ ]:
pd.read_sql("cuotas", engine, "id")

In [ ]:
import src.logic.settlements as settlements
from src.database.models import TipoLiquidacionEnum  # noqa: F401
reload(settlements)

fecha = "2026/07/29"
NewSett = settlements.SettlementManager()
df_rec, df_s_rec = NewSett.obtain_settlement_of_transferred_quota(4, "CLIENTE ID", fecha, fecha_vencimiento_hasta="2026/06/28")
df = NewSett.settlements_s_resource(df_s_rec)
# NewSett.execute_settlements(fecha)

df.groupby(["tipo_cobranza"])[["capital", "interes", "iva"]].sum().map("$ {:,.2f}".format)